# Чистка треков Suno в Kaggle (в фоне)

Та же нейросеть, что в Colab (Mel-RoFormer Denoise), но работает **сама, без тебя у экрана**.

**Один раз настроить (справа, панель Settings / Session options):**
- **Accelerator → GPU T4 x2**
- **Internet → On**

**Каждый выпуск:**
1. На Google Диске открыть архив для всех по ссылке (как для бота) и скопировать ссылку.
2. Вставить её в ячейку 1 ниже, в `TRACKS_LINK`.
3. Вверху **Save Version → Save & Run All (Commit) → Save**. Можно закрывать страницу.
4. Через 20–50 минут открыть этот блокнот → вкладка **Output** → скачать `… (clean).zip`.
5. Положить на Диск, открыть по ссылке, отправить ссылку боту как обычно.

⚠️ Первый запуск — пробный: в Kaggle этот блокнот ещё не запускался.

In [ ]:
# 1. Настройки
# Ссылка на архив с треками на Google Диске (доступ «всем, у кого есть ссылка»)
TRACKS_LINK = ""
# Сила чистки: "aggressive" — смелее с шумом поверх тарелок; "normal" — бережнее
STRENGTH = "aggressive"
# Скорость: "fast" — примерно вдвое быстрее; "quality" — медленнее, максимум качества
SPEED = "fast"


In [ ]:
# 2. Ставим нейросеть и проверяем видеокарту (пара минут)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'audio-separator[gpu]', 'gdown'], check=True)
import torch
assert torch.cuda.is_available(), 'Нет видеокарты: справа Settings → Accelerator → GPU T4 x2'
print('Видеокарта:', torch.cuda.get_device_name(0))


In [ ]:
# 3. Скачиваем архив с Диска
import os, glob, shutil, gdown
assert TRACKS_LINK.strip(), 'Вставь ссылку на архив в ячейку 1 (TRACKS_LINK)'
shutil.rmtree('/tmp/src', ignore_errors=True); os.makedirs('/tmp/src')
src = gdown.download(TRACKS_LINK.strip(), output='/tmp/src/', fuzzy=True, quiet=False)
assert src and src.lower().endswith('.zip'), f'Скачалось не то (нужен .zip): {src}. Проверь доступ по ссылке.'
print('Архив:', os.path.basename(src), round(os.path.getsize(src) / 1e6), 'МБ')


In [ ]:
# 4. Чистим каждый трек
import zipfile
from audio_separator.separator import Separator

MODELS = {
    'aggressive': 'denoise_mel_band_roformer_aufr33_aggr_sdr_27.9768.ckpt',
    'normal': 'denoise_mel_band_roformer_aufr33_sdr_27.9959.ckpt',
}
work = '/tmp/work'; shutil.rmtree(work, ignore_errors=True)
os.makedirs(f'{work}/in'); os.makedirs(f'{work}/out')
with zipfile.ZipFile(src) as z:
    # порядок файлов в архиве важен сборщику — сохраняем его
    order = [n for n in z.namelist() if n.lower().endswith(('.wav', '.mp3', '.flac')) and not n.startswith('__MACOSX')]
    z.extractall(f'{work}/in')
print('Треков:', len(order))

sep = Separator(output_dir=f'{work}/out', output_format='WAV', model_file_dir='/tmp/models',
                mdxc_params={'segment_size': 256, 'overlap': 2 if SPEED == 'fast' else 4, 'batch_size': 1})
sep.load_model(model_filename=MODELS[STRENGTH])

clean = {}
for i, name in enumerate(order, 1):
    outs = sep.separate(os.path.join(f'{work}/in', name))
    dry = [o for o in outs if '(dry)' in o or '(Dry)' in o][0]
    clean[name] = os.path.join(f'{work}/out', os.path.basename(dry))
    for o in outs:
        if o != dry:
            try: os.remove(os.path.join(f'{work}/out', os.path.basename(o)))
            except FileNotFoundError: pass
    print(f'{i}/{len(order)} готов: {os.path.basename(name)}')


In [ ]:
# 5. Кладём чистый архив во вкладку Output
base = os.path.splitext(os.path.basename(src))[0]
dst = f'/kaggle/working/{base} (clean).zip'
for old in glob.glob('/kaggle/working/*.zip'):
    os.remove(old)
with zipfile.ZipFile(dst, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in order:  # те же имена и тот же порядок, что в исходном архиве
        z.write(clean[name], arcname=os.path.basename(name).rsplit('.', 1)[0] + '.wav')
shutil.rmtree(work, ignore_errors=True)
print('Готово:', os.path.basename(dst), round(os.path.getsize(dst) / 1e6), 'МБ')
print('Забери во вкладке Output, положи на Диск и отправь ссылку боту.')
